# Box Office Bomb Data Pipeline Assignment

**Total Marks: 20**

This notebook implements an end-to-end pipeline to:
1. Scrape box office bomb data from Wikipedia
2. Validate & clean data using Pydantic
3. Enrich with OMDb API data
4. Perform consistency checks
5. Create final categorized dataset

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel,Field, field_validator, ValidationError
from typing import Optional
import re
import time
from pathlib import Path

## Task 1: Scrape the "Bombs" Table (4 Marks)

Extract raw data from the Wikipedia HTML file:
- Film Title (with symbols like § and †)
- Year
- Net production budget (may contain ranges like "$100–160")
- Estimated loss (Nominal column)

Note:
- You need to extract the entire raw string with the symbols, references, etc along with the titles.
- You must handle the nested headers in the Wikipedia table (Budget and Loss columns have sub-headers).

In [ ]:
def scrape_box_office_bombs():
    """
    Scrape the Biggest box-office bombs table from Wikipedia.
    Returns a list of dictionaries with raw extracted data.
    """

     # TODO: Load the HTML file
    url = "https://en.wikipedia.org/wiki/List_of_biggest_box-office_bombs"

    # Add headers to mimic a real browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/143.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers)
    html = response.text
  # TODO: Parse with BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")

  # TODO: Find the main table - it's wikitable sortable with caption "Biggest box-office bombs"

    target_table = None
    for table in soup.find_all("table", class_="wikitable"):
        caption = table.find("caption")
        if caption and "Biggest box-office bombs" in caption.get_text():
            target_table = table
            break

    if not target_table:
        return []
    table_body = target_table.find("tbody")
    table_rows = table_body.find_all("tr")

    raw_data = []
  # TODO: Iterate through rows (skip headers) and extract: 'raw_title', 'raw_year', 'raw_budget', 'raw_loss'.
  # Store extracted data in raw_data. Example entry: {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$10.4'}

    for row in table_rows:
      title_cell = row.find("th", scope="row")
      data = row.find_all("td")
      raw_row={}
      # Only process rows with actual movie titles and enough columns
      if title_cell and len(data) >= 4:
          raw_row = {
              "raw_title": title_cell.get_text(strip=True),
              "raw_year": data[0].get_text(strip=True),
              "raw_budget": data[1].get_text(strip=True),
              "raw_loss": data[3].get_text(strip=True)
          }
          raw_data.append(raw_row)

    return raw_data

In [ ]:
# Test the scraping function
raw_movies = scrape_box_office_bombs()
print(f"Scraped {len(raw_movies)} movies")
print("\nLast 15 raw entries:")
for i, movie in enumerate(raw_movies[-15:]):
    print(f"\n{i+1}. {movie}")

Scraped 139 movies

Last 15 raw entries:

1. {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$85'}

2. {'raw_title': 'Transformers: The Last Knight', 'raw_year': '2017', 'raw_budget': '$217–260', 'raw_loss': '$100+'}

3. {'raw_title': 'Treasure Planet', 'raw_year': '2002', 'raw_budget': '$140', 'raw_loss': '$85'}

4. {'raw_title': 'Tron: Ares†', 'raw_year': '2025', 'raw_budget': '$220', 'raw_loss': '$132.7'}

5. {'raw_title': 'Turning Red§', 'raw_year': '2022', 'raw_budget': '$175', 'raw_loss': '$173'}

6. {'raw_title': 'Valerian and the City of a Thousand Planets', 'raw_year': '2017', 'raw_budget': '$177.2–180', 'raw_loss': '$82'}

7. {'raw_title': 'West Side Story', 'raw_year': '2021', 'raw_budget': '$100', 'raw_loss': '$104'}

8. {'raw_title': 'Wild Wild West', 'raw_year': '1999', 'raw_budget': '$170', 'raw_loss': '$66.2'}

9. {'raw_title': 'Windtalkers', 'raw_year': '2002', 'raw_budget': '$115–120', 'raw_loss': '$76–81'}

10. {'raw_title': 'Wis

## Task 2: Pydantic Data Parsing & Validation (6 Marks)

Create a Pydantic model that:
- Cleans titles (removes §, †, and footnotes like [nb 2], [1])
- Parses numeric values (handles ranges, currency symbols)
- Validates year as integer

In [ ]:
class MovieData(BaseModel):
    """
    Pydantic model for validating and cleaning movie data.
    """
    # TODO: Define the 4 required fields with their types:
    # title (str), year (int), budget_millions (float), loss_millions (float)

    title: str= Field(...,min_length=1,description="Movie title")
    year: int=Field(...,description="releasing year")
    budget_millions: float=Field(...,description="budget in millions")
    loss_millions: float=Field(...,description="loss in millions")



    @field_validator('title', mode='before')
    @classmethod
    def clean_title(cls, v):
        # Hint: Use .replace() for symbols and re.sub() for footnotes
        """
        Remove footnote markers and special symbols from title.
        - Remove § (streaming symbol)
        - Remove † (currently playing symbol)
        - Remove footnotes like [nb 2], [1], etc.
        """
        # TODO: Implement title cleaning logic
        if(pd.isnull(v) or v=="" or v=="N/A"):
            return None
        cleaned_title=re.sub(r'\[.*?\]','',v)
        v=cleaned_title.replace("§","").replace("†","").strip()
        return v


    @field_validator('year', mode='before')
    @classmethod
    def validate_year(cls, v):
        """
        Ensure year is a valid integer.
        Remove any extra characters and convert to int.
        """
        if(pd.isnull(v) or v=="" or v=="N/A"):
            return None
        # TODO: Clean string (remove non-digits) and convert to integer
        v=re.sub(r'[^0-9]','',v)
        if v == "":
          return None
        v = int(v)
        return v

    @field_validator('budget_millions', mode='before')
    @classmethod
    def parse_budget(cls, v):
        """
        Parse budget value:
        - Strip $ and other currency symbols
        - Handle ranges (e.g., "100–160") by calculating average
        - Remove reference tags
        """
        if(pd.isnull(v) or v =="" or v =="N/A"):
            return None
        # TODO: Call your helper method _parse_numeric_value
        v=cls._parse_numeric_value(v)
        return v


    @field_validator('loss_millions', mode='before')
    @classmethod
    def parse_loss(cls, v):
        """
        Parse loss value with same logic as budget.
        """
        if(pd.isnull(v) or v =="" or v =="N/A"):
            return None
        # TODO: Call your helper method _parse_numeric_value
        v=cls._parse_numeric_value(v)
        return v

    @staticmethod
    def _parse_numeric_value(v):
        """
        Helper method to parse numeric values with ranges.
        """
        # TODO: Implement logic to:
        # 1. Clean string (remove $, reference tags)
        # 2. Check for ranges (hyphen or en-dash) and calculate average
        # 3. Return a float
        v = re.sub(r'[^0-9–\-]', '', v)
        if v == "":
          return None
        parts = v.replace("–", "-").split("-")
        numbers = [float(p) for p in parts if p.strip()]

        v = sum(numbers) / len(numbers)
        return v



In [ ]:
# Validate and clean the raw data
validated_movies = []
failed_validations = []

for raw_movie in raw_movies:
    try:
        movie = MovieData(
            title=raw_movie['raw_title'],
            year=raw_movie['raw_year'],
            budget_millions=raw_movie['raw_budget'],
            loss_millions=raw_movie['raw_loss']
        )
        validated_movies.append(movie)
    except ValidationError as e:
        failed_validations.append({
            'raw_data': raw_movie,
            'error': str(e)
        })
        print(f"Validation failed for {raw_movie['raw_title']}: {e}")

print(f"\n{'='*60}")
print(f"Validation Results:")
print(f"Successfully validated: {len(validated_movies)} movies")
print(f"Failed validations: {len(failed_validations)} movies")
print(f"{'='*60}")

# Show first 3 validated movies
print("\nLast 15 validated movies:")
for i, movie in enumerate(validated_movies[-15:]):
    print(f"\n{i+1}. {movie.model_dump()}")


Validation Results:
Successfully validated: 139 movies
Failed validations: 0 movies

Last 15 validated movies:

1. {'title': 'Town & Country', 'year': 2001, 'budget_millions': 90.0, 'loss_millions': 85.0}

2. {'title': 'Transformers: The Last Knight', 'year': 2017, 'budget_millions': 238.5, 'loss_millions': 100.0}

3. {'title': 'Treasure Planet', 'year': 2002, 'budget_millions': 140.0, 'loss_millions': 85.0}

4. {'title': 'Tron: Ares', 'year': 2025, 'budget_millions': 220.0, 'loss_millions': 1327.0}

5. {'title': 'Turning Red', 'year': 2022, 'budget_millions': 175.0, 'loss_millions': 173.0}

6. {'title': 'Valerian and the City of a Thousand Planets', 'year': 2017, 'budget_millions': 976.0, 'loss_millions': 82.0}

7. {'title': 'West Side Story', 'year': 2021, 'budget_millions': 100.0, 'loss_millions': 104.0}

8. {'title': 'Wild Wild West', 'year': 1999, 'budget_millions': 170.0, 'loss_millions': 662.0}

9. {'title': 'Windtalkers', 'year': 2002, 'budget_millions': 117.5, 'loss_millions'

## Task 3: Enrich with OMDb Data (4 Marks)

Query OMDb API for each movie to get:
- Plot
- Metascore
- IMDb Rating
- Director
- Language

Handle API failures (Response='False') or missing fields ('N/A') gracefully by storing them as None. Do not delete the row.

In [ ]:
OMDB_API_KEY = "69403b52"
OMDB_BASE_URL = "http://www.omdbapi.com/"

def clean_value(value):
    if value == "N/A" or value == "" or pd.isna(value):
        return None
    return value

def query_omdb(title: str, year: int) -> dict:
    params = {
        "apikey": OMDB_API_KEY,
        "t": title,
        "y": year,
        "type": "movie"
    }

    try:
        response = requests.get(OMDB_BASE_URL, params=params)

        if response.ok:
            data = response.json()

            if data.get("Response") == "False":
                return {
                    "omdb_year": None,
                    "Plot": None,
                    "Metascore": None,
                    "imdbRating": None,
                    "Director": None,
                    "Language": None
                }

            result = {
                "omdb_year": clean_value(data.get("Year")),
                "Plot": clean_value(data.get("Plot")),
                "Metascore": clean_value(data.get("Metascore")),
                "imdbRating": clean_value(data.get("imdbRating")),
                "Director": clean_value(data.get("Director")),
                "Language": clean_value(data.get("Language"))
            }
            return result

        else:
            return {
                "omdb_year": None,
                "Plot": None,
                "Metascore": None,
                "imdbRating": None,
                "Director": None,
                "Language": None
            }

    except Exception as e:
        print(f"Error: {e}")
        return {
            "omdb_year": None,
            "Plot": None,
            "Metascore": None,
            "imdbRating": None,
            "Director": None,
            "Language": None
        }

In [ ]:
# Enrich each validated movie with OMDb data
enriched_data = []
print("Querying OMDb API...")
for movie in validated_movies:
  omdb_result=query_omdb(movie.title,movie.year)
  raw_omdb_year = omdb_result.get("omdb_year")
  try:
        omdb_result["omdb_year"] = int(raw_omdb_year) if raw_omdb_year is not None else None
  except ValueError:
        omdb_result["omdb_year"] = None
  combined={"title":movie.title,"year":movie.year,**omdb_result}
  enriched_data.append(combined)
# TODO: Loop through validated_movies, call query_omdb, and merge data
print(f"\nEnriched {len(enriched_data)} movies with OMDb data")
# Show sample enriched data
print("\nFirst enriched entry:")
print(enriched_data[0])

Querying OMDb API...

Enriched 139 movies with OMDb data

First enriched entry:
{'title': 'The 13th Warrior', 'year': 1999, 'omdb_year': 1999, 'Plot': 'A man, having fallen in love with the wrong woman, is sent by the sultan himself on a diplomatic mission to a distant land as an ambassador. Stopping at a Viking village port to restock on supplies, he finds himself unwittingly em...', 'Metascore': '42', 'imdbRating': '6.6', 'Director': 'John McTiernan', 'Language': 'English, Latin, Swedish, Norse, Old, Danish, Arabic'}


## Task 4: Data Consistency Check (2 Marks)

Compare Wikipedia year with OMDb year:
- "Verified": Years match (±1 year tolerance)
- "Mismatch": Years differ by >1
- "Not Found": OMDb returned no data

In [ ]:
def determine_match_status(wiki_year: int, omdb_year: Optional[int]) -> str:
    """
    Determine the match status between Wikipedia and OMDb years.
    Returns "Verified", "Mismatch", or "Not Found".
    """
    if omdb_year is None:
        return "Not Found"
    try:
        omdb_year_numeric = int(omdb_year)
    except ValueError:
        return "Not Found" # If omdb_year can't be converted to int

    if abs(wiki_year - omdb_year_numeric) <= 1:
        return "Verified"
    else:
        return "Mismatch"

# Apply this function to your enriched_data list and add match_status to each entry
for movie in enriched_data:
  movie["match_status"]=determine_match_status(movie["year"],movie["omdb_year"])

# Show match status distribution
df_temp = pd.DataFrame(enriched_data)
print("Match Status Distribution:")
print(df_temp['match_status'].value_counts())

# Show some mismatches if any
mismatches = df_temp[df_temp['match_status'] == 'Mismatch']
if len(mismatches) > 0:
    print(f"\nSample Mismatches (showing up to 5):")
    print(mismatches[['title', 'year', 'omdb_year', 'match_status']].head())

Match Status Distribution:
match_status
Verified     137
Not Found      2
Name: count, dtype: int64


## Task 5: Final Dataset & Categorization (4 Marks)

Create final DataFrame with:
- Loss_Category based on estimated loss:
  - "Catastrophic": Loss ≥ \$100M

  - "Severe": Loss between \$50M and \$100M

  - "Moderate": Loss < \$50M
- Save to `box_office_failures.csv`

Required columns: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status

In [ ]:
def categorize_loss(loss_millions: float) -> str:
    """
    Categorize the financial loss into tiers.
    """
    # TODO: Implement categorization logic
    if loss_millions >= 100:
        return "Catastrophic"
    elif loss_millions >= 50:
        return "Severe"
    else:
        return "Moderate"
for i, movie in enumerate(validated_movies):
    enriched_data[i]["Budget_Millions"] = movie.budget_millions
    enriched_data[i]["Loss_Millions"] = movie.loss_millions
for movie in enriched_data:
  movie["match_status"] = determine_match_status(movie["year"], movie["omdb_year"])
df_final = pd.DataFrame(enriched_data)
df_final["Loss_Category"] = df_final["Loss_Millions"].apply(categorize_loss)
df_final = df_final.rename(columns={
    "title": "Title",
    "year": "Year",
    "Director": "Director",
    "Language": "Language",
    "imdbRating": "IMDb_Rating",
    "match_status": "Match_Status"
})
df_final = df_final[
    [
        "Title",
        "Year",
        "Director",
        "Language",
        "Budget_Millions",
        "Loss_Millions",
        "Loss_Category",
        "IMDb_Rating",
        "Metascore",
        "Match_Status"
    ]
]
df_final.to_csv("box_office_failures.csv", index=False)
print("Final Dataset Summary:")
print(f"Total movies: {len(df_final)}")
print(f"\nLoss Category Distribution:")
print(df_final['Loss_Category'].value_counts())
print(f"\nBasic Statistics:")
print(df_final[['Budget_Millions', 'Loss_Millions', 'IMDb_Rating', 'Metascore']].describe())

# Display first few rows
print(f"\nFirst 10 rows of final dataset:")
df_final.head(10)

Final Dataset Summary:
Total movies: 139

Loss Category Distribution:
Loss_Category
Severe          73
Catastrophic    66
Name: count, dtype: int64

Basic Statistics:
       Budget_Millions  Loss_Millions
count       139.000000     139.000000
mean        191.956835     229.960432
std         283.965587     316.838785
min          17.000000      60.000000
25%          86.500000      77.750000
50%         137.000000      97.500000
75%         187.500000     142.000000
max        2637.000000    1748.000000

First 10 rows of final dataset:


,Title,Year,Director,Language,Budget_Millions,Loss_Millions,Loss_Category,IMDb_Rating,Metascore,Match_Status
0,The 13th Warrior,1999,John McTiernan,"English, Latin, Swedish, Norse, Old, Danish, A...",130.0,99.0,Severe,6.6,42,Verified
1,The 355,2022,Simon Kinberg,"English, Chinese, Spanish, French, German, Ara...",57.5,93.0,Severe,5.6,40,Verified
2,47 Ronin,2013,Carl Rinsch,"English, Japanese",200.0,96.0,Severe,6.2,28,Verified
3,The Adventures of Baron Munchausen,1988,Terry Gilliam,English,466.0,385.0,Catastrophic,7.1,69,Verified
4,The Adventures of Pluto Nash,2002,Ron Underwood,English,100.0,96.0,Severe,3.9,12,Verified
5,The Adventures of Rocky & Bullwinkle,2000,Des McAnuff,English,531.0,635.0,Catastrophic,4.3,36,Verified
6,The Alamo,2004,John Lee Hancock,"English, Spanish",107.0,94.0,Severe,6.1,47,Verified
7,Alexander,2004,Oliver Stone,English,155.0,71.0,Severe,5.6,40,Verified
8,Ali,2001,Michael Mann,"English, French, Swahili",107.0,63.0,Severe,6.7,65,Verified
9,Allied,2016,Robert Zemeckis,"English, French, German, Arabic",85.0,82.5,Severe,7.1,60,Verified


In [ ]:
# Save to CSV
output_path = 'box_office_failures.csv'

# TODO: Save DataFrame to CSV

print(f"✓ Dataset saved to: {output_path}")
print(f"✓ Total records: {len(df_final)}")

✓ Dataset saved to: box_office_failures.csv
✓ Total records: 139


## Additional Analysis (Optional)

In [ ]:
# Show some interesting insights
print("Top 10 Biggest Box Office Bombs (by loss):")
print(df_final.nlargest(10, 'Loss_Millions')[['Title', 'Year', 'Director', 'Loss_Millions', 'Loss_Category']])

print("\n" + "="*60)
print("\nMovies with Lowest IMDb Ratings:")
df_final['IMDb_Rating'] = pd.to_numeric(df_final['IMDb_Rating'], errors='coerce')
lowest_rated = df_final[df_final['IMDb_Rating'].notna()].nsmallest(5, 'IMDb_Rating')
print(lowest_rated[['Title', 'Year', 'Director', 'IMDb_Rating', 'Metascore', 'Loss_Millions']])

print("\n" + "="*60)
print("\nAverage Loss by Category:")
print(df_final.groupby('Loss_Category')['Loss_Millions'].agg(['mean', 'count']))

print("\n" + "="*60)
print("\nTop 5 Most Common Directors in Box Office Bombs:")
print(df_final['Director'].value_counts().head())

Top 10 Biggest Box Office Bombs (by loss):
                                Title  Year          Director  Loss_Millions  \
86                     Mortal Engines  2018  Christian Rivers         1748.0   
68                Joker: Folie à Deux  2024     Todd Phillips         1443.0   
127                        Tron: Ares  2025   Joachim Rønning         1327.0   
136                 A Wrinkle in Time  2018      Ava DuVernay         1306.0   
44            Furiosa: A Mad Max Saga  2024     George Miller         1196.0   
45                         Gemini Man  2019           Ang Lee         1111.0   
82                       Missing Link  2019      Chris Butler         1013.0   
101                        Robin Hood  2018     Otto Bathurst          837.0   
72   King Arthur: Legend of the Sword  2017       Guy Ritchie          822.0   
102                            Sahara  2005      Breck Eisner          784.0   

    Loss_Category  
86   Catastrophic  
68   Catastrophic  
127  Catastrophi